# Predicción probabilística de fraude con tarjetas

**Informe técnico final — LDS1071 · Kaggle Playground Series S6E11**

La variable observada es binaria, $Y\in\{0,1\}$, pero Kaggle solicita la probabilidad $\hat p(x)=P(Y=1\mid X=x)$. La métrica oficial es ROC AUC; convertir a decisiones de alerta es una etapa operativa distinta.

El reporte sigue el diseño de los experimentos: **EDA → representación y agrupamientos → selección e hiperparámetros → ensambles → desempeño y costo → incertidumbre → alertas y conclusiones**. Se utilizan cinco pliegues externos estratificados; los ajustes y la selección de características no usan observaciones del pliegue externo de validación. Los resultados anteriores se reutilizan por defecto sin reentrenamiento.

## Etapa 0. Configuración, fuentes y trazabilidad

Trabajamos desde la raíz de `Credit-Fraud`. En Windows: `conda activate lds1071-fraud` y seleccione el kernel `Python (LDS1071 Fraud)`. El ZIP oficial `playground-series-s6e11.zip` debe estar en la raíz o los CSV en `data/`. Las figuras nuevas se guardan bajo `outputs/informe_final_es/figuras/` y **no se sobrescriben** los resultados originales.

In [ ]:
from __future__ import annotations
import os, zipfile, json, io, warnings, time, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, log_loss, brier_score_loss

SEMILLA = 42
N_PARTICIONES = 5
MUESTRA_GRAFICAS = 20_000
CORRER_BUSQUEDA_DE_EJEMPLO = False   # True: ejecuta una nueva GridSearchCV, no sustituye el registro histórico.
RECALCULAR_BOOTSTRAP = False         # True: recalcula 600 réplicas si se dispone del OOF privado.

RAIZ = Path.cwd()
if not (RAIZ / 'credit_card_fraud.ipynb').exists() and (RAIZ / 'final_project' / 'credit_card_fraud.ipynb').exists():
    RAIZ = RAIZ / 'final_project'
if not (RAIZ / 'credit_card_fraud.ipynb').exists():
    # Permitir ejecutar la versión descargada aislada junto a sus archivos de soporte.
    RAIZ = Path.cwd()
DATOS_AGREGADOS = RAIZ / 'experiments' / 'experiment016' / 'resultados'
if not DATOS_AGREGADOS.is_dir() and (RAIZ / 'resultados').is_dir():
    DATOS_AGREGADOS = RAIZ / 'resultados'
SALIDA = RAIZ / 'outputs' / 'informe_final_es'
FIGURAS = SALIDA / 'figuras'
FIGURAS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='dark', context='notebook')
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':150, 'axes.titlesize':12, 'axes.titleweight':'semibold'})
COLORES = sns.color_palette('dark')

def cerrar_figura(fig, nombre):
    fig.tight_layout()
    fig.savefig(FIGURAS / f'{nombre}.png', bbox_inches='tight', dpi=150)
    plt.show(); plt.close(fig)

DICCIONARIO = json.loads((DATOS_AGREGADOS/'diccionario_variables_es.json').read_text(encoding='utf-8')) if (DATOS_AGREGADOS/'diccionario_variables_es.json').exists() else {}
def etiqueta(col):
    return DICCIONARIO.get(col, str(col).replace('_',' ').capitalize())

print('Ubicación del proyecto:', RAIZ.resolve())
print('Modo: reutilizar resultados previos; no se reentrenarán modelos automáticamente.')
print('Versiones: Python', sys.version.split()[0], '| pandas', pd.__version__)

def anotaciones_matriz(m):
    total=np.sum(m)
    return np.array([[f'{int(valor):,}\n({valor/total:.2%})' for valor in fila] for fila in m],dtype=object)

## Etapa 1. Datos e integridad

El conjunto contiene **699 960** transacciones etiquetadas y **299 984** de prueba, con **24 variables explicativas**: 13 numéricas, 5 categóricas y 6 lógicas. La columna `id` identifica la operación y **no se usa como predictor**. En el conjunto de entrenamiento la clase minoritaria contiene 11 689 fraudes (aproximadamente 1.67 %). El problema combina desequilibrio de clases y riesgos de relaciones no lineales.

In [ ]:
def leer_csv(nombre):
    for ruta in [RAIZ/'data'/nombre, RAIZ/nombre, Path('/kaggle/input/playground-series-s6e11')/nombre]:
        if ruta.is_file():
            return pd.read_csv(ruta)
    for ruta in [RAIZ/'playground-series-s6e11.zip', RAIZ/'data'/'playground-series-s6e11.zip', RAIZ.parent/'playground-series-s6e11.zip']:
        if ruta.is_file():
            with zipfile.ZipFile(ruta) as archivo:
                with archivo.open(nombre) as fichero:
                    return pd.read_csv(fichero)
    raise FileNotFoundError(f'No se encontró {nombre}. Coloque el ZIP original de Kaggle en la raíz.')

entrenamiento = leer_csv('train.csv')
prueba = leer_csv('test.csv')
plantilla_envio = leer_csv('sample_submission.csv')
OBJETIVO, IDENTIFICADOR = 'is_fraud', 'id'
assert OBJETIVO not in prueba and plantilla_envio[IDENTIFICADOR].equals(prueba[IDENTIFICADOR])
assert entrenamiento[IDENTIFICADOR].is_unique and prueba[IDENTIFICADOR].is_unique
assert entrenamiento[OBJETIVO].notna().all() and set(entrenamiento[OBJETIVO].unique()) == {0.,1.}
variables = [c for c in entrenamiento.columns if c not in [OBJETIVO, IDENTIFICADOR]]
assert set(variables) == set(prueba.columns) - {IDENTIFICADOR}
numericas = entrenamiento[variables].select_dtypes(include='number').columns.tolist()
categoricas = entrenamiento[variables].select_dtypes(include=['object','category']).columns.tolist()
logicas = entrenamiento[variables].select_dtypes(include='bool').columns.tolist()
y = entrenamiento[OBJETIVO].astype(int).to_numpy()
assert len(variables)==24 and len(numericas)==13 and len(categoricas)==5 and len(logicas)==6
print(f'Entrenamiento: {len(entrenamiento):,} | Prueba: {len(prueba):,} | Variables: {len(variables)}')
print('Fraude:',int(y.sum()),f'({y.mean():.2%})')
display(entrenamiento.head(3))
resumen_variables = pd.DataFrame({'variable':variables,'nombre_en_español':[etiqueta(c) for c in variables],
 'tipo':[str(entrenamiento[c].dtype) for c in variables], 'faltantes':[int(entrenamiento[c].isna().sum()) for c in variables]})
display(resumen_variables)

## Etapa 2. Análisis exploratorio de datos (EDA)

La exploración utiliza totales completos para prevalencia, tasas por categoría e interacciones; en gráficos con muchos puntos se toma una muestra aleatoria de tamaño declarado, para evitar saturación. **No se elimina automáticamente ninguna observación extrema**: una transacción inusual puede contener información relevante para detectar fraude.

### 2.1 Desbalance, valores faltantes y distribución numérica

In [ ]:
muestra = entrenamiento.sample(n=min(MUESTRA_GRAFICAS,len(entrenamiento)), random_state=SEMILLA).copy()
fig, ejes = plt.subplots(1,2,figsize=(13,4.7))
conteos = entrenamiento[OBJETIVO].value_counts().reindex([0.,1.],fill_value=0)
sns.barplot(x=['Legítimas','Fraudulentas'],y=conteos.values,hue=['Legítimas','Fraudulentas'],
            palette=[COLORES[0],COLORES[3]],legend=False,ax=ejes[0])
for k,v in enumerate(conteos): ejes[0].text(k,v*1.01,f'{v:,} ({v/len(entrenamiento):.2%})',ha='center',fontsize=10)
ejes[0].set(title='Distribución observada de clases',xlabel='Clase',ylabel='Número de transacciones',ylim=(0,conteos.max()*1.14))
faltantes=entrenamiento[variables].isna().mean()*100
sns.barplot(x=faltantes.values,y=[etiqueta(c) for c in faltantes.index],color=COLORES[0],ax=ejes[1])
ejes[1].set(title='Datos faltantes por variable',xlabel='Porcentaje de valores faltantes',ylabel='Variable',xlim=(0,max(1,float(faltantes.max())*1.12)))
cerrar_figura(fig,'01_desbalance_y_faltantes')

cols=[c for c in ['amount_usd','hours_since_last_txn','distance_from_home_km','velocity_score','merchant_risk_score','account_balance_usd'] if c in muestra]
fig,ejes=plt.subplots(2,3,figsize=(16,8.8))
for ax,col in zip(ejes.ravel(),cols):
    sns.histplot(data=muestra,x=col,bins=42,color=COLORES[0],ax=ax)
    if col in ('amount_usd','account_balance_usd') and (muestra[col] > 0).all(): ax.set_xscale('log')
    ax.set(title=etiqueta(col),xlabel=etiqueta(col),ylabel='Transacciones (muestra)')
for ax in ejes.ravel()[len(cols):]: ax.axis('off')
fig.suptitle('Distribuciones de variables numéricas (muestra aleatoria)',y=1.01,fontweight='bold')
cerrar_figura(fig,'02_distribuciones_numericas')

### 2.2 Dependencias entre predictores y tasas de fraude por categorías

La correlación de Pearson mide asociación lineal; Spearman evalúa ordenamientos monótonos. La ausencia de correlación alta **no** descarta interacciones ni dependencias no lineales. Las tasas por categoría se calculan sobre el **conjunto de entrenamiento completo**, no sobre la muestra empleada para visualizar.

In [ ]:
matriz=entrenamiento[numericas+logicas].astype(float)
fig,ax=plt.subplots(figsize=(12,10))
sns.heatmap(matriz.corr(method='spearman'),cmap='vlag',center=0,vmin=-1,vmax=1,
            xticklabels=[etiqueta(c) for c in matriz.columns],
            yticklabels=[etiqueta(c) for c in matriz.columns],ax=ax,
            cbar_kws={'label':'Correlación de Spearman'})
ax.set_title('Asociación monótona entre variables numéricas y lógicas')
cerrar_figura(fig,'03_correlacion_spearman')

fig,ejes=plt.subplots(1,2,figsize=(15.3,5.8))
for ax,col in zip(ejes,['merchant_category','channel']):
    g=entrenamiento.groupby(col,dropna=False)[OBJETIVO].agg(['mean','size']).sort_values('mean',ascending=False)
    g['nombre']=[f'{cat} (n={n:,})' for cat,n in zip(g.index,g['size'])]
    sns.barplot(data=g,x='mean',y='nombre',color=COLORES[0],ax=ax)
    ax.axvline(np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(xlabel='Proporción observada de fraude',ylabel='',title=f'Riesgo por {etiqueta(col).lower()}')
    ax.legend(title='Referencia')
fig.suptitle('Tasas de fraude según categoría y canal (entrenamiento completo)',y=1.01,fontweight='bold')
cerrar_figura(fig,'04_fraude_por_categorias')

### 2.3 No linealidad e interacciones

Una transformación que retenga el 90 % de la varianza no necesariamente preservará los contrastes asociados al fraude. Antes de modelar evaluamos tasas de fraude por cuantiles para identificar posibles umbrales y relaciones no monótonas. Las tasas marginales no son efectos causales ni sustituyen al ajuste multivariante.

In [ ]:
analizar=['amount_usd','hours_since_last_txn','velocity_score','merchant_risk_score']
fig,ejes=plt.subplots(2,2,figsize=(14.4,9))
for ax,col in zip(ejes.ravel(),analizar):
    intervalos=pd.qcut(entrenamiento[col],q=12,duplicates='drop')
    tabla=entrenamiento.groupby(intervalos,observed=True)[OBJETIVO].agg(['mean','size']).reset_index()
    sns.lineplot(x=np.arange(1,len(tabla)+1),y=100*tabla['mean'],marker='o',color=COLORES[0],ax=ax)
    ax.axhline(100*np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(title=etiqueta(col),xlabel='Intervalo por cuantiles (de menor a mayor)',ylabel='Operaciones fraudulentas (%)')
    ax.legend()
fig.suptitle('Relaciones marginales no lineales con el fraude',fontweight='bold',y=1.01)
cerrar_figura(fig,'05_tasas_por_cuantiles')

pares=[('merchant_risk_score','velocity_score'),('amount_usd','distance_from_home_km')]
fig,ejes=plt.subplots(1,2,figsize=(15,6.1))
for ax,(a,b) in zip(ejes,pares):
    ga=pd.qcut(entrenamiento[a],q=5,labels=False,duplicates='drop')
    gb=pd.qcut(entrenamiento[b],q=5,labels=False,duplicates='drop')
    t=entrenamiento.groupby([ga.rename('a'),gb.rename('b')])[OBJETIVO].mean().unstack()*100
    sns.heatmap(t,annot=True,fmt='.2f',cmap='crest',ax=ax,cbar_kws={'label':'Fraude observado (%)'})
    ax.set(xlabel=f'{etiqueta(b)} (quintil)',ylabel=f'{etiqueta(a)} (quintil)',title='Tasa conjunta por quintiles')
fig.suptitle('Interacciones bivariantes en el conjunto completo',fontweight='bold',y=1.01)
cerrar_figura(fig,'06_interacciones_quintiles')

### 2.4 Patrones temporales, valores atípicos y anomalías

Una hora del día es una variable periódica: 23:00 y 00:00 están próximas pese a tener valores numéricos distantes. Se consideran transformaciones trigonométricas como posibilidad de ingeniería de variables. Para los extremos se emplean cercas de 3×IQR únicamente como **diagnóstico descriptivo**, junto con un detector no supervisado (Isolation Forest) sobre una muestra.

In [ ]:
fig,ejes=plt.subplots(1,2,figsize=(14,4.8))
for ax,col in zip(ejes,['time_of_day_hour','day_of_week']):
    tasa=entrenamiento.groupby(col)[OBJETIVO].agg(['mean','size'])
    sns.barplot(x=tasa.index,y=100*tasa['mean'],color=COLORES[0],ax=ax)
    ax.axhline(100*np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(xlabel=etiqueta(col),ylabel='Porcentaje de fraude',title='Variación temporal del riesgo')
    ax.legend()
cerrar_figura(fig,'07_variacion_temporal')

extremos=[]
for c in numericas:
    q1,q3=entrenamiento[c].quantile([.25,.75]); iq=q3-q1
    if iq==0:continue
    flag=(entrenamiento[c]<q1-3*iq)|(entrenamiento[c]>q3+3*iq)
    extremos.append({'Variable':etiqueta(c),'Proporción extrema':float(flag.mean()),'Tasa de fraude entre extremos':float(entrenamiento.loc[flag,OBJETIVO].mean()) if flag.any() else np.nan})
tabla_extremos=pd.DataFrame(extremos).sort_values('Proporción extrema',ascending=False)
fig,ax=plt.subplots(figsize=(12,6))
sns.barplot(data=tabla_extremos.head(9),x='Proporción extrema',y='Variable',color=COLORES[3],ax=ax)
ax.set(title='Valores fuera de las cercas conservadoras de 3×IQR',xlabel='Proporción de operaciones',ylabel='Variable')
cerrar_figura(fig,'08_extremos_univariados')

if len(muestra)>500:
    base_anom=muestra[numericas+logicas].astype(float).replace([np.inf,-np.inf],np.nan)
    base_anom=base_anom.fillna(base_anom.median())
    scaler_anom=RobustScaler().fit_transform(base_anom)
    iso=IsolationForest(n_estimators=90,max_samples=256,contamination=0.02,random_state=SEMILLA,n_jobs=-1)
    score=-iso.fit(scaler_anom).decision_function(scaler_anom)
    m=muestra[[OBJETIVO]].copy();m['índice de anomalía']=score
    m['clase']=m[OBJETIVO].map({0.:'Legítima',1.:'Fraudulenta'})
    fig,ax=plt.subplots(figsize=(11,4.7))
    sns.histplot(data=m,x='índice de anomalía',hue='clase',stat='density',common_norm=False,
                 bins=40,element='step',fill=False,ax=ax,palette={'Legítima':COLORES[0],'Fraudulenta':COLORES[3]})
    ax.set(title='Anomalía multivariante: distribución según clase (muestra)',xlabel='Puntuación de anomalía (mayor = más atípica)',ylabel='Densidad por clase')
    cerrar_figura(fig,'09_anomalias_isolation_forest')

### 2.5 Comparación entre entrenamiento y prueba

Como las etiquetas de la prueba son desconocidas, solo se comparan las distribuciones de los predictores. Una diferencia de distribuciones entre `train` y `test` podría comprometer la generalización; **no** implica que exista una etiqueta de fraude disponible en la prueba.

In [ ]:
variables_traspaso=['amount_usd','velocity_score','merchant_risk_score']
fig,ejes=plt.subplots(1,3,figsize=(16,4.7))
for ax,c in zip(ejes,variables_traspaso):
    a=entrenamiento[c].sample(n=min(12000,len(entrenamiento)),random_state=SEMILLA).reset_index(drop=True)
    b=prueba[c].sample(n=min(12000,len(prueba)),random_state=SEMILLA).reset_index(drop=True)
    sns.ecdfplot(a,ax=ax,color=COLORES[0],label='Entrenamiento')
    sns.ecdfplot(b,ax=ax,color=COLORES[1],label='Prueba')
    ax.set(title=etiqueta(c),xlabel=etiqueta(c),ylabel='Proporción acumulada')
    ax.legend()
fig.suptitle('Comparación no supervisada de las distribuciones',fontweight='bold',y=1.01)
cerrar_figura(fig,'10_desplazamiento_entrenamiento_prueba')

## 3. Selección de la representación de los datos

La representación también es una elección del modelo. Se evalúan variables originales, bases *spline*, PCA/SVD con un objetivo del 90 % de varianza, K-means (distancias y pertenencias suaves) y DBSCAN (densidad, núcleos y ruido). **Son columnas derivadas, no transacciones sintéticas**.

Para los datos centrados, $X_c=U\Sigma V^\top$. Preservar varianza no es lo mismo que preservar información discriminante sobre fraude. Cada transformación predictiva debe ajustarse exclusivamente dentro del entrenamiento de cada pliegue.

In [ ]:
# PCA solamente ilustrativa sobre una muestra de entrenamiento: no interviene en ninguna predicción OOF.
X_visual=muestra[variables]
preprocesador_visual=ColumnTransformer([
    ('num',Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]),numericas+logicas),
    ('cat',Pipeline([('imputar',SimpleImputer(strategy='most_frequent')),
                     ('codificar',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categoricas),
],sparse_threshold=0)
M=preprocesador_visual.fit_transform(X_visual)
pca_v=PCA(n_components=.90,svd_solver='full',random_state=SEMILLA).fit(M)
pca_total=PCA(n_components=min(30,M.shape[1]),svd_solver='full',random_state=SEMILLA).fit(M)
fig,ejes=plt.subplots(1,2,figsize=(14.5,4.8))
ejes[0].plot(np.arange(1,len(pca_total.explained_variance_ratio_)+1),pca_total.explained_variance_ratio_.cumsum(),marker='o',markersize=3,color=COLORES[0])
ejes[0].axhline(.90,ls='--',color=COLORES[3],label='90 % de varianza')
ejes[0].set(xlabel='Número de componentes',ylabel='Varianza acumulada',title='PCA ilustrativa: varianza explicada')
ejes[0].legend()
XY=PCA(n_components=2,random_state=SEMILLA).fit_transform(M)
seleccion=np.random.default_rng(SEMILLA).choice(len(XY),size=min(5000,len(XY)),replace=False)
clases=muestra.iloc[seleccion][OBJETIVO].values.astype(int)
for val,color,nombre in [(0,COLORES[0],'Legítima'),(1,COLORES[3],'Fraudulenta')]:
    mask=clases==val
    ejes[1].scatter(XY[seleccion[mask],0],XY[seleccion[mask],1],s=8,alpha=.48,color=color,label=nombre,rasterized=True)
ejes[1].set(xlabel='Primera componente',ylabel='Segunda componente',title='Proyección PCA en dos dimensiones')
ejes[1].legend(title='Clase observada')
fig.suptitle('Reducción dimensional para explorar la estructura de los predictores',fontweight='bold',y=1.01)
cerrar_figura(fig,'11_pca_90_varianza')
print('PCA ilustrativa: componentes para 90 % de varianza:',pca_v.n_components_,'de',M.shape[1])

### 3.1 t-SNE: proyección exploratoria de vecindades

Se proyectan datos preprocesados en dos dimensiones mediante t-SNE. La representación de los fraudes se enriquece exclusivamente para que sean visibles en la gráfica; **no** es la prevalencia real y las coordenadas no se usan para clasificar. PCA y la geometría de clustering son transformaciones diferentes, ajustadas dentro de entrenamiento para prevenir fuga de información.

In [ ]:
from sklearn.manifold import TSNE
generador_tsne=np.random.default_rng(SEMILLA)
pos=np.flatnonzero(muestra[OBJETIVO].to_numpy()==1)
neg=np.flatnonzero(muestra[OBJETIVO].to_numpy()==0)
idx=np.r_[generador_tsne.choice(neg,size=min(1500,len(neg)),replace=False),
          generador_tsne.choice(pos,size=min(270,len(pos)),replace=False)]
coords=TSNE(n_components=2,perplexity=35,init='pca',learning_rate='auto',max_iter=800,random_state=SEMILLA).fit_transform(M[idx])
clases=muestra.iloc[idx][OBJETIVO].to_numpy()
fig,ax=plt.subplots(figsize=(9,5.6))
for clase,color,nombre in [(0,COLORES[0],'Legítima'),(1,COLORES[3],'Fraudulenta')]:
    mask=clases==clase
    ax.scatter(coords[mask,0],coords[mask,1],color=color,s=15,alpha=.5,label=f'{nombre} (n={mask.sum():,})',rasterized=True)
ax.set(title='t-SNE: vecindades locales (muestra enriquecida)',xlabel='Coordenada t-SNE 1',ylabel='Coordenada t-SNE 2')
ax.legend(title='Clase observada')
cerrar_figura(fig,'11b_tsne')

### 3.2 K-means y DBSCAN: hiperparámetros de los datos

K-means compara inercia y silueta para (k\in\{3,5,8,12\}). La inercia decrece normalmente con (k), de modo que no basta minimizarla. También se evaluó un contraste controlado con **cinco grupos** y similitud al centro. DBSCAN explora radios de vecindad, número de grupos y fracción de ruido; para nuevas observaciones se calculan distancias a núcleos aprendidos solo en entrenamiento.

La representación —original, K-means, DBSCAN o ambas— debe entrar en la validación interna como una **opción de preparación del modelo**, no añadirse una vez evaluados los resultados OOF.

In [ ]:
ruta_diag=DATOS_AGREGADOS/'experiment013_cluster_diagnostics.csv'
if ruta_diag.exists():
    diagnosticos=pd.read_csv(ruta_diag)
    diag_k=diagnosticos[(diagnosticos['algorithm']=='KMeans')]
    diag_db=diagnosticos[(diagnosticos['algorithm']=='DBSCAN')]
    fig,ejes=plt.subplots(1,3,figsize=(16.5,4.7))
    sns.lineplot(data=diag_k,x='k',y='silhouette',hue='fold',marker='o',palette='dark',ax=ejes[0])
    ejes[0].set(title='K-means: índice de silueta',xlabel='Número de clusters (k)',ylabel='Silueta')
    ejes[0].legend(title='Pliegue',fontsize=8)
    sns.lineplot(data=diag_k,x='k',y='inertia',hue='fold',marker='o',palette='dark',ax=ejes[1])
    ejes[1].set(title='K-means: inercia',xlabel='Número de clusters (k)',ylabel='Inercia total')
    ejes[1].legend(title='Pliegue',fontsize=8)
    sns.lineplot(data=diag_db,x='param',y='noise_fraction',hue='fold',marker='o',palette='dark',ax=ejes[2])
    ejes[2].set(title='DBSCAN: proporción identificada como ruido',xlabel='Radio de vecindad (ε)',ylabel='Proporción de ruido')
    ejes[2].legend(title='Pliegue',fontsize=8)
    fig.suptitle('Diagnóstico no supervisado dentro de los entrenamientos por pliegue',y=1.02,fontweight='bold')
    cerrar_figura(fig,'12b_seleccion_clusters')
else:
    print('El archivo de diagnósticos de clusters no está disponible.')

### 3.2 K-means y DBSCAN como hiperparámetros de representación

K-means utiliza la inercia $\sum_i\|z_i-\mu_{c(i)}\|^2$ y el índice de silueta para comparar $k\in\{3,5,8,12\}$. Se estudió también un control de exactamente cinco grupos. DBSCAN compara radios $\varepsilon$, número de clusters y proporción de ruido. Las características para observaciones futuras se obtienen a partir de los núcleos identificados en entrenamiento.

La opción de representación —original, K-means, DBSCAN o ambas— puede formar parte de la búsqueda **dentro de la validación interna**. No se recalculan clusters una sola vez sobre todas las transacciones.

In [ ]:
resumen_ablasiones=[]
for e,modelo in [(13,'XGBoost individual'),(14,'Regresión logística'),(15,'Regresión logística + XGBoost')]:
    p=DATOS_AGREGADOS/f'experiment{e:03d}_summary.csv'
    if p.is_file():
        d=pd.read_csv(p)
        d['modelo']=modelo;d['experimento']=e
        resumen_ablasiones.append(d)
if resumen_ablasiones:
    ablasiones=pd.concat(resumen_ablasiones,ignore_index=True)
    campos=['modelo','variant','roc_auc','delta_auc_vs_baseline']
    display(ablasiones[[c for c in campos if c in ablasiones.columns]].round(6))
    fig,ax=plt.subplots(figsize=(12,6.6))
    tabla=ablasiones[ablasiones['variant']!='baseline'].copy()
    trad={'kmeans':'K-means','dbscan':'DBSCAN','combined':'Ambos','combined_lr_only':'Ambos, solo en la logística'}
    tabla['Variables nuevas']=tabla['variant'].map(trad).fillna(tabla['variant'])
    tabla['Modelo']=tabla['modelo']
    sns.barplot(data=tabla,y='Modelo',x='delta_auc_vs_baseline',hue='Variables nuevas',ax=ax)
    ax.axvline(0,color='black',lw=1)
    ax.set(title='Cambio de ROC AUC tras incorporar variables de agrupamiento',xlabel='Diferencia respecto del modelo sin agrupamientos',ylabel='Familia evaluada')
    ax.legend(title='Características añadidas',bbox_to_anchor=(1.01,1),loc='upper left')
    cerrar_figura(fig,'12_ablasiones_agrupamiento')
else:
    print('No se encontraron tablas agregadas de los experimentos 013–015.')

## 4. Comparación de familias e hiperparámetros

La regresión logística modela efectos lineales sobre el logit; las bases spline aportan flexibilidad, PCA controla dimensión, el bosque aleatorio agrega árboles y XGBoost/CatBoost ajustan conjuntos de árboles potenciados. El híbrido utiliza una regresión logística como base y XGBoost para aprender correcciones.

La grilla del booster incluye $\mathrm{max\_depth}\in\{3,5,7,10,15,30\}$, $\eta\in\{0.04,0.08\}$ y $\mathrm{min\_child\_weight}\in\{5,10,15\}$. La evaluación externa usa cinco pliegues estratificados y la selección se realiza en el entrenamiento externo.

In [ ]:
ruta_exp17=RAIZ/'outputs'/'experiment017'/'full_699960'
ruta_comp=RAIZ/'outputs'/'experiment017'/'depth30'/'summary_expanded.csv'
if not ruta_comp.is_file():
    ruta_comp=RAIZ/'experiments'/'experiment017'/'results'/'summary_kmeans5.csv'
if ruta_comp.is_file():
    resultados_017=pd.read_csv(ruta_comp)
    if 'variant' not in resultados_017 and 'modelo' in resultados_017:
        resultados_017=resultados_017.rename(columns={'modelo':'variant'})
    nombres_017={'original_24_variables':'XGBoost original','kmeans5_fijo':'XGBoost + K-means fijo',
                 'kmeans5_grilla_30':'XGBoost + K-means, 30 configuraciones',
                 'kmeans5_grilla_36':'XGBoost + K-means, 36 configuraciones'}
    resultados_017['Configuración']=resultados_017['variant'].map(nombres_017).fillna(resultados_017['variant'])
    display(resultados_017[['Configuración','roc_auc','precision_promedio','perdida_logaritmica','brier']].sort_values('roc_auc',ascending=False).round(6))
    tabla=resultados_017.sort_values('roc_auc')
    fig,ax=plt.subplots(figsize=(12,4.5))
    pos=np.arange(len(tabla))
    ax.scatter(tabla['roc_auc'],pos,s=65,color=COLORES[0])
    ax.set_yticks(pos,tabla['Configuración'])
    mn,mx=tabla['roc_auc'].min(),tabla['roc_auc'].max()
    ax.set_xlim(mn-.0004,mx+.0007)
    for j,v in enumerate(tabla['roc_auc']):
        ax.annotate(f'{v:.6f}',(v,j),xytext=(7,0),textcoords='offset points',va='center')
    ax.set(title='XGBoost: predictores originales frente a cinco grupos K-means',
           xlabel='ROC AUC OOF (eje ampliado)',ylabel='')
    cerrar_figura(fig,'13c_xgboost_kmeans5_comparacion')
    elegidos=RAIZ/'experiments'/'experiment017'/'results'/'selected_params_36.csv'
    if elegidos.is_file():display(pd.read_csv(elegidos))
else:
    print('No se encontraron los agregados del experimento017: consulte su README.')

### 3.5 Conclusión de ingeniería de variables

Los resultados de clustering y PCA se incorporan a la tabla de especificaciones **antes de elegir el modelo final**. La mejora parcial de la logística con distancias no demuestra una ganancia después de la corrección XGBoost. Las alternativas quedan explícitas para búsqueda anidada, sin declarar superioridad por una diferencia pequeña.

In [ ]:
fig,ax=plt.subplots(figsize=(13.5,5.5))
resumen=pd.DataFrame([
    ['XGBoost individual',0.9457023626,'Sin nuevas variables'],
    ['XGBoost + DBSCAN',0.9457050194,'DBSCAN'],
    ['Logística individual',0.9413568422,'Sin nuevas variables'],
    ['Logística + K-means y DBSCAN',0.9415591273,'Ambos'],
    ['Logística + XGBoost',0.9461023509,'Sin nuevas variables'],
    ['Logística con clusters + XGBoost original',0.9461606241,'Clusters solo en logística'],
],columns=['Modelo','ROC AUC','Ingeniería adicional'])
sns.barplot(data=resumen,x='ROC AUC',y='Modelo',hue='Ingeniería adicional',ax=ax)
ax.set(title='Ablaciones completas de agrupamientos: experimentos 013–015',xlabel='ROC AUC OOF',ylabel='Arquitectura')
ax.legend(title='Variables nuevas',loc='lower right')
cerrar_figura(fig,'18_ablasiones_resumen_final')

### 4.3 Búsqueda de XGBoost después de la regresión logística

Se evalúan 36 configuraciones del booster. Para el XGBoost individual se dispone de GridSearchCV. Para la corrección de logits, un evaluador exhaustivo con caché **por pliegue interno** es necesario: XGBoost recibe los logits de la regresión logística como base_margin **durante el entrenamiento y al predecir**.

Las transformaciones de agrupamiento y las predicciones logísticas se ajustan dentro de cada partición interna. Aplicar directamente GridSearchCV al estimador XGBoost sin un envoltorio que propague base_margin a predict_proba no evaluaría el modelo híbrido correctamente.

In [ ]:
# Registro de configuraciones de la fase original; los nombres legibles se usan en las tablas.
ESPECIFICACIONES = [
 ('logit_C0p1','Regresión logística','C=0.1'),
 ('logit_C1','Regresión logística','C=1'),
 ('logit_balanced_C1','Logística con ponderación de clases','C=1; clases equilibradas'),
 ('logit_spline_C0p1','Logística con splines','C=0.1'),
 ('logit_spline_C1','Logística con splines','C=1'),
 ('pca90_logit_C0p1','Logística con PCA','90 % de varianza; C=0.1'),
 ('pca90_logit_C1','Logística con PCA','90 % de varianza; C=1'),
 ('rf_depth12_leaf10','Bosque aleatorio','profundidad 12; hoja mínima 10'),
 ('rf_depth18_leaf5','Bosque aleatorio','profundidad 18; hoja mínima 5'),
 ('rf_depthNone_leaf15','Bosque aleatorio','profundidad libre; hoja mínima 15'),
 ('xgb_depth3_lr08','XGBoost individual','profundidad 3; tasa 0.08'),
 ('xgb_depth5_lr05','XGBoost individual','profundidad 5; tasa 0.05; 350 árboles'),
 ('xgb_depth6_lr04','XGBoost individual','profundidad 6; tasa 0.04'),
 ('catboost_depth5_lr06','CatBoost','profundidad 5; tasa 0.06'),
 ('catboost_depth7_lr04','CatBoost','profundidad 7; tasa 0.04'),
 ('offset_xgb_depth2_lr05','Logística corregida con XGBoost','profundidad de corrección 2; tasa 0.05'),
 ('offset_xgb_depth3_lr05','Logística corregida con XGBoost','profundidad de corrección 3; tasa 0.05; 240 árboles'),
 ('xgb_nested_grid','XGBoost ajustado con GridSearchCV','búsqueda interna de hiperparámetros'),
 ('catboost_nested_grid','CatBoost ajustado con GridSearchCV','búsqueda interna de hiperparámetros'),
]
tabla_modelos=pd.DataFrame(ESPECIFICACIONES,columns=['Clave del experimento','Modelo','Configuración'])
display(tabla_modelos)

### 4.1 Comprobaciones particulares de la regresión logística

La regresión logística supone una respuesta binaria y una forma aproximadamente lineal para el **logit**, no para la probabilidad. No exige normalidad de las variables explicativas. La regularización L2 reduce inestabilidad por colinealidad, pero no sustituye una evaluación de separación, dependencia entre observaciones y calibración. Los diagnósticos siguientes distinguen hechos observables de supuestos que no pueden verificarse con el conjunto entregado.

In [ ]:
supuestos=pd.DataFrame([
    ['Respuesta binaria','Verificado','La etiqueta toma los valores 0 y 1.'],
    ['Predictores observados / faltantes','Verificado','Se auditan los faltantes y la imputación se ajusta por pliegue.'],
    ['Linealidad en el logit','No garantizada','Se detectaron patrones no lineales; se contrastaron splines y árboles.'],
    ['Independencia por cliente','No verificable','No se proporciona un identificador de cliente longitudinal.'],
    ['Colinealidad / redundancia','Mitigada','Categorías con nivel de referencia y regularización L2.'],
    ['Separación completa','No demostrada ausente','Una revisión univariada no descarta separación multivariada.'],
    ['Tamaño de muestra','Adecuado para empezar','Hay 11 689 fraudes; importa también la rareza de combinaciones de categorías.'],
    ['Calibración de probabilidades','Se evalúa con OOF','Pérdida logarítmica, Brier y curvas de calibración.'],
],columns=['Condición','Situación','Evidencia y limitación'])
display(supuestos)

### 4.3 GridSearchCV y búsqueda anidada

La grilla completa de XGBoost contiene 36 combinaciones. El ajuste XGBoost individual usa búsqueda de scikit-learn. Para la **corrección de logits**, la evaluación debe pasar `base_margin` tanto en entrenamiento como en predicción. Un recorrido exhaustivo con geometría y logits cacheados **por split interno** evita fuga y reproduce el mismo objetivo que una búsqueda de grilla con un envoltorio apropiado.

In [ ]:
def preprocesador_numerico_y_categorico(escalar=True):
    pasos=[('imputar',SimpleImputer(strategy='median'))]
    if escalar: pasos.append(('escalar',StandardScaler()))
    return ColumnTransformer([
        ('numeros',Pipeline(pasos),numericas+logicas),
        ('categorias',Pipeline([('imputar',SimpleImputer(strategy='most_frequent')),
                               ('codificar',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categoricas),
    ],sparse_threshold=0)

GRILLA_XGBOOST={'modelo__max_depth':[3,5,7,10,15,30], 'modelo__learning_rate':[0.04,0.08], 'modelo__min_child_weight':[5,10,15]}
GRILLA_CATBOOST={'modelo__depth':[4,6], 'modelo__learning_rate':[0.04,0.08], 'modelo__l2_leaf_reg':[3.0,7.0]}
print('XGBoost:', len(GRILLA_XGBOOST['modelo__max_depth'])*len(GRILLA_XGBOOST['modelo__learning_rate'])*len(GRILLA_XGBOOST['modelo__min_child_weight']),'combinaciones')
print('CatBoost:', len(GRILLA_CATBOOST['modelo__depth'])*len(GRILLA_CATBOOST['modelo__learning_rate'])*len(GRILLA_CATBOOST['modelo__l2_leaf_reg']),'combinaciones')

if CORRER_BUSQUEDA_DE_EJEMPLO:
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier
    datos_b, _=train_test_split(entrenamiento,train_size=min(12000,len(entrenamiento)-1),stratify=y,random_state=SEMILLA)
    X_b=datos_b[variables]; y_b=datos_b[OBJETIVO].astype(int)
    candidatos={
        'XGBoost': (XGBClassifier(n_estimators=180,objective='binary:logistic',tree_method='hist',device='cpu',n_jobs=-1,random_state=SEMILLA), GRILLA_XGBOOST),
        'CatBoost': (CatBoostClassifier(iterations=180,task_type='CPU',verbose=False,thread_count=-1,random_seed=SEMILLA),GRILLA_CATBOOST),
    }
    registros_grillas=[]
    for nombre,(estimador,grilla) in candidatos.items():
        pipe=Pipeline([('preprocesar',preprocesador_numerico_y_categorico(escalar=False)),('modelo',estimador)])
        cv_interna=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEMILLA)
        buscador=GridSearchCV(pipe,grilla,cv=cv_interna,scoring='roc_auc',n_jobs=1,refit=True,return_train_score=True)
        buscador.fit(X_b,y_b)
        registros_grillas.append({'Modelo':nombre,'Mejor ROC AUC interna':buscador.best_score_,'Parámetros':buscador.best_params_})
    display(pd.DataFrame(registros_grillas))
else:
    print('Búsqueda desactivada: los resultados de la comparación se recuperan de la ejecución histórica.')

### 4.4 Tres operaciones separadas: representación, selección y evaluación

Primero se construyen variables **solo** con entrenamiento interno. Luego se buscan opciones de representación e hiperparámetros usando AUC interna. Por último se reentrena la configuración seleccionada sobre todo el entrenamiento externo y se predice su fold retenido. La nueva búsqueda de correcciones utiliza tres folds internos sobre una muestra estratificada de 36 000 filas de cada entrenamiento externo; el ajuste final sí usa todas sus filas.

In [ ]:
from sklearn.base import clone

def evaluar_busqueda_anidada(X_total,y_total,estimador_base,grilla,max_filas_busqueda=20_000):
    """Devuelve OOF y parámetros ganadores por fold sin mirar las etiquetas validadas."""
    externo=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEMILLA)
    oof=np.full(len(y_total),np.nan,dtype=float)
    ganadores=[]
    y_total=np.asarray(y_total,dtype=int)
    for plegue,(ent,val) in enumerate(externo.split(X_total,y_total),start=1):
        x_ent=X_total.iloc[ent];y_ent=y_total[ent]
        if len(ent)>max_filas_busqueda:
            x_bus,_,y_bus,_=train_test_split(x_ent,y_ent,train_size=max_filas_busqueda,
                                             stratify=y_ent,random_state=SEMILLA+plegue)
        else:
            x_bus,y_bus=x_ent,y_ent
        interno=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEMILLA+100+plegue)
        busqueda=GridSearchCV(clone(estimador_base),grilla,cv=interno,scoring='roc_auc',
                              refit=True,n_jobs=1,error_score='raise')
        busqueda.fit(x_bus,y_bus)
        final=clone(estimador_base).set_params(**busqueda.best_params_)
        final.fit(x_ent,y_ent)
        oof[val]=final.predict_proba(X_total.iloc[val])[:,1]
        ganadores.append({'Pliegue externo':plegue,'Mejor ROC AUC interno':busqueda.best_score_,
                          'Parámetros elegidos':busqueda.best_params_,
                          'ROC AUC externo':roc_auc_score(y_total[val],oof[val])})
    assert np.isfinite(oof).all()
    return oof,pd.DataFrame(ganadores)

ruta_grid=RAIZ/'outputs'/'nested_gridsearch_cv_results.csv'
if ruta_grid.is_file():
    reg=pd.read_csv(ruta_grid)
    reg['Familia ajustada']=reg['candidate'].map({'xgb_nested_grid':'XGBoost ajustado','catboost_nested_grid':'CatBoost ajustado'}).fillna(reg['candidate'])
    fig,ax=plt.subplots(figsize=(11,5.3))
    sns.boxplot(data=reg,x='Familia ajustada',y='mean_test_roc_auc',color=COLORES[0],ax=ax)
    ax.set(title='Distribución del ROC AUC de las búsquedas internas',xlabel='Familia de modelos',ylabel='ROC AUC interno de validación')
    cerrar_figura(fig,'13b_busqueda_hiperparametros')
    display(reg[['Familia ajustada','outer_fold','mean_test_roc_auc','std_test_roc_auc','rank_test_roc_auc']].head(12).round(6))
else:
    print('Sin resultados originales de búsqueda anidada en esta ubicación; no se representan puntuaciones no disponibles.')

### 4.5 Comparación de candidatos individuales

La figura utiliza un **eje ampliado** y valores exactos de ROC AUC OOF para hacer visibles diferencias pequeñas. No se confunden los candidatos con hiperparámetros fijos con las búsquedas internas de GridSearchCV.

In [ ]:
archivos_candidatos=[RAIZ/'outputs'/'candidate_metrics.csv',RAIZ.parent/'outputs'/'candidate_metrics.csv']
archivo_candidatos=next((p for p in archivos_candidatos if p.is_file()),None)
NOMBRES_CANDIDATO=dict((a,f'{b} · {c}') for a,b,c in ESPECIFICACIONES)
if archivo_candidatos:
    candidatos_originales=pd.read_csv(archivo_candidatos)
    if {'candidate','roc_auc'}.issubset(candidatos_originales.columns):
        candidatos_originales['Modelo legible']=candidatos_originales['candidate'].map(NOMBRES_CANDIDATO).fillna(candidatos_originales['candidate'])
        display(candidatos_originales[['Modelo legible','roc_auc','average_precision','log_loss','brier']].sort_values('roc_auc',ascending=False).round(6))
        top=candidatos_originales.sort_values('roc_auc').tail(19).copy()
        top['Nombre corto']=top['Modelo legible'].str.replace(' · búsqueda interna de hiperparámetros',' (grilla)',regex=False)
        fig,ax=plt.subplots(figsize=(13.8,max(7.5,.46*len(top)+1.8)))
        posiciones=np.arange(len(top))
        ax.scatter(top['roc_auc'],posiciones,color=COLORES[0],s=55,zorder=4)
        mejor=float(top['roc_auc'].max())
        ax.axvline(mejor,color=COLORES[3],ls='--',lw=1.3,label=f'Mejor valor: {mejor:.6f}')
        menor=float(top['roc_auc'].min())
        margen=max((mejor-menor)*.065,.0006)
        ax.set_xlim(menor-margen,mejor+2.2*margen)
        ax.set_yticks(posiciones,top['Nombre corto'])
        for yy,valor in zip(posiciones,top['roc_auc']):
            ax.annotate(f'{valor:.6f}',(valor,yy),xytext=(6,0),textcoords='offset points',va='center',fontsize=8)
        ax.set(title='Comparación de candidatos por ROC AUC OOF',xlabel='ROC AUC fuera de partición (eje ampliado)',ylabel='Configuración')
        ax.grid(axis='x',alpha=.35)
        ax.legend(loc='lower right',fontsize=9)
        cerrar_figura(fig,'13_comparacion_candidatos')
else:
    print('Sin candidate_metrics.csv: no se inventan resultados de candidatos.')

### 4.6 Búsqueda de la corrección XGBoost después de la logística

La búsqueda anidada posterior incorpora **cuatro representaciones de la etapa logística** (originales, K-means, DBSCAN y ambas) y 36 combinaciones de profundidad, tasa de aprendizaje y tamaño de hoja del booster. XGBoost recibe los logits como `base_margin` durante el ajuste **y** la predicción. Las transformaciones se refitan en cada split interno. El XGBoost corrector usa los predictores originales para aislar el efecto de mejorar la etapa logística. Cinco folds externos entrenan con todas sus observaciones; la búsqueda interna usa 36 000 registros de entrenamiento externo por fold.

In [ ]:
RUTA_HIBRIDO=RAIZ/'experiments'/'experiment018'/'resultados'
if not RUTA_HIBRIDO.exists():RUTA_HIBRIDO=RAIZ/'outputs'/'experiment018'/'full_699960'
archivo_hibrido=RUTA_HIBRIDO/'summary.csv'
if archivo_hibrido.is_file():
    res_hibrido=pd.read_csv(archivo_hibrido).sort_values('roc_auc',ascending=False)
    mapa={'baseline':'Datos originales','kmeans':'Logística con K-means','dbscan':'Logística con DBSCAN','combined':'Logística con ambos'}
    res_hibrido['Representación']=res_hibrido['variant'].map(mapa)
    display(res_hibrido[['Representación','roc_auc','average_precision','log_loss','brier']].round(6))
    fig,ax=plt.subplots(figsize=(11,5))
    sns.scatterplot(data=res_hibrido,x='roc_auc',y='Representación',s=130,color=COLORES[0],ax=ax)
    for j,v in enumerate(res_hibrido.roc_auc):ax.annotate(f'{v:.6f}',(v,j),xytext=(6,0),textcoords='offset points')
    ax.set(title='Corrección XGBoost: búsqueda de representación e hiperparámetros',xlabel='ROC AUC OOF (eje ampliado)',ylabel='')
    cerrar_figura(fig,'13e_busqueda_hibrida_con_clustering')
else:
    print('No se encontraron resultados completos de la búsqueda anidada; no se inventan AUC.')

## 5. Ensambles y corrección probabilística

Se contrastan contra el mejor XGBoost individual: (1) promedio de probabilidades (**blending**), (2) logística corregida con XGBoost usando `base_margin`, (3) XGBoost de regresión sobre residuos probabilísticos (y-\hat p_{LR}), y (4) **stacking** con bosque aleatorio entrenado sobre columnas OOF de modelos base. Solo la segunda estrategia corrige los **logits** con una pérdida binaria de boosting. La validación del stacking no fue completamente anidada, de modo que su puntuación es exploratoria.

In [ ]:
# Los resultados agregados se conservan con su origen y nunca se confunden con nuevas ejecuciones.
p=DATOS_AGREGADOS/'five_way_comparison.csv'
if not p.exists(): raise FileNotFoundError(f'No se encontró {p}; conserve el directorio resultados incluido con el notebook.')
comparacion=pd.read_csv(p)
print('Columnas originales:',list(comparacion.columns))
display(comparacion.head())

### 5.1 Especificación implementable de los ensambles

Las siguientes funciones muestran las **operaciones exactas** que diferencian las arquitecturas; están separadas del reporte de la ejecución histórica para evitar que un nuevo ajuste cambie las comparaciones verificadas. En la corrección de logits, `base_margin` contiene los logits de la logística, no los residuos; XGBoost calcula los gradientes de la pérdida binaria durante el boosting. En el residual explícito se cambia el objetivo por $y-\hat p_{\rm LR}$. En el stacking solo entran las probabilidades OOF de los modelos base.

In [ ]:
def media_de_probabilidades(*predicciones):
    matriz=np.column_stack([np.asarray(p,dtype=float) for p in predicciones])
    if not (np.isfinite(matriz).all() and np.min(matriz)>=0 and np.max(matriz)<=1):
        raise ValueError('Los ensambles deben recibir probabilidades válidas.')
    return matriz.mean(axis=1)

def entrenar_correccion_logits(X_ent,y_ent,X_val,regresion_logistica,arboles=240,dispositivo='cpu'):
    """Modelo secuencial original: corregir logits con pérdida logística."""
    from xgboost import XGBClassifier
    regresion_logistica.fit(X_ent,y_ent)
    logit_ent=np.asarray(regresion_logistica.decision_function(X_ent),dtype=np.float32)
    booster=XGBClassifier(n_estimators=arboles,max_depth=3,learning_rate=.05,
                          min_child_weight=5,subsample=.85,colsample_bytree=.85,
                          tree_method='hist',device=dispositivo,n_jobs=-1,
                          objective='binary:logistic',eval_metric='logloss',random_state=SEMILLA)
    booster.fit(X_ent,y_ent,base_margin=logit_ent)
    logit_val=np.asarray(regresion_logistica.decision_function(X_val),dtype=np.float32)
    return booster.predict_proba(X_val,base_margin=logit_val)[:,1]

def construir_objetivo_residual(y_ent,p_logistica_ent):
    """Objetivo explícito: residuos de probabilidades, no logits."""
    return np.asarray(y_ent,dtype=float)-np.asarray(p_logistica_ent,dtype=float)

def corregir_probabilidades(p_logistica,p_residuo_estimada):
    """Se recorta la suma para mantenerla dentro de [0,1]."""
    return np.clip(np.asarray(p_logistica)+np.asarray(p_residuo_estimada),0,1)

def matriz_para_stacking(diccionario_oof,nombres):
    """El bosque de nivel 2 solo usa una columna de probabilidades OOF por base."""
    return np.column_stack([np.asarray(diccionario_oof[n],dtype=float) for n in nombres])

print('Definiciones disponibles. No se reentrenaron ensambles en esta sección.')

### 5.2 Comparación de las cinco estrategias por ROC AUC

Se representa únicamente la métrica oficial de Kaggle, **ROC AUC OOF**, con seis decimales y escala horizontal ampliada. Las métricas auxiliares se conservan en la tabla. El mejor XGBoost individual y el modelo híbrido están muy próximos; se comprueba su diferencia mediante bootstrap emparejado.

In [ ]:
traduccion_estrategias={
 'best_single':'XGBoost individual',
 'ensemble1_blend':'Promedio de probabilidades',
 'ensemble2_logit_offset':'Regresión logística corregida con XGBoost',
 'ensemble3_explicit_residual':'Corrección explícita de residuos',
 'ensemble4_rf_stacking':'Stacking con bosque aleatorio'}
col_modelo='strategy' if 'strategy' in comparacion else 'name'
comparacion['Estrategia']=comparacion[col_modelo].map(traduccion_estrategias).fillna(comparacion[col_modelo])
metricas=['roc_auc','average_precision','log_loss','brier']
for metrica in metricas: assert metrica in comparacion
orden=comparacion.sort_values('roc_auc',ascending=False)
display(orden[['Estrategia']+metricas].round(6))
tabla=orden.iloc[::-1]
fig,ax=plt.subplots(figsize=(12.8,5.1))
sns.barplot(data=tabla,x='roc_auc',y='Estrategia',color=COLORES[0],ax=ax)
valores=tabla['roc_auc'].to_numpy(dtype=float)
minimo=float(valores.min());maximo=float(valores.max())
ax.set_xlim(minimo-.0009,maximo+.0014)
for yidx,v in enumerate(valores):
    ax.annotate(f'{v:.6f}',xy=(v,yidx),xytext=(6,0),textcoords='offset points',va='center',fontsize=9)
ax.axvline(float(orden.loc[orden[col_modelo]=='best_single','roc_auc'].iloc[0]),
           ls='--',color=COLORES[3],lw=1.25,label='Mejor modelo individual')
ax.set(title='Mejor modelo individual frente a cuatro estrategias de ensamble',
       xlabel='ROC AUC fuera de partición (escala ampliada)',ylabel='')
ax.legend(loc='lower right',fontsize=9)
cerrar_figura(fig,'14_comparacion_cinco_estrategias')

### 5.2 Curvas ROC, precisión–exhaustividad y calibración (si existen OOF privadas)

Estas curvas se recalculan con las etiquetas reales y las probabilidades **guardadas** de cada estrategia. Por seguridad, los vectores OOF individuales no están en este paquete público. Si no están presentes en su equipo, se omiten estas gráficas y se conservan las métricas verificadas sin interpolarlas artificialmente.

In [ ]:
def localizar_npz():
    ubicaciones=[RAIZ/'outputs'/'ensemble_extension'/'five_way_oof_and_test_predictions.npz',
                 RAIZ.parent/'outputs'/'ensemble_extension'/'five_way_oof_and_test_predictions.npz',
                 RAIZ/'ensemble_extension'/'five_way_oof_and_test_predictions.npz']
    for u in ubicaciones:
        if u.is_file(): return ('archivo',u)
    for u in [RAIZ/'ensemble_extension.zip',RAIZ.parent/'ensemble_extension.zip']:
        if u.is_file():
            with zipfile.ZipFile(u) as z:
                claves=[a for a in z.namelist() if a.endswith('five_way_oof_and_test_predictions.npz')]
                if claves:return ('zip',(u,claves[0]))
    return None
fuente_oof=localizar_npz()
resultados_oof={}
if fuente_oof:
    tipo,ref=fuente_oof
    if tipo=='archivo':
        almacenamiento=np.load(ref,allow_pickle=False)
    else:
        ruta,nombre=ref
        with zipfile.ZipFile(ruta) as z:
            almacenamiento=np.load(io.BytesIO(z.read(nombre)),allow_pickle=False)
    try:
        for nombre in traduccion_estrategias:
            clave_oof=f'{nombre}__oof';clave_prueba=f'{nombre}__test'
            if clave_oof not in almacenamiento or clave_prueba not in almacenamiento:continue
            po=np.asarray(almacenamiento[clave_oof],dtype=np.float64)
            pt=np.asarray(almacenamiento[clave_prueba],dtype=np.float64)
            if po.shape!=(len(y),) or pt.shape!=(len(prueba),):
                raise ValueError(f'Longitud incorrecta de {nombre}: puede ser un orden o conjunto distintos')
            if not(np.isfinite(po).all() and np.isfinite(pt).all() and 0<=po.min()<=po.max()<=1 and 0<=pt.min()<=pt.max()<=1):
                raise ValueError('Probabilidades no válidas o no finitas: '+nombre)
            resultados_oof[nombre]={'oof':po,'test':pt}
    finally:
        almacenamiento.close()
    for fila in comparacion.itertuples():
        codigo=getattr(fila,col_modelo)
        if codigo in resultados_oof:
            observado=roc_auc_score(y,resultados_oof[codigo]['oof'])
            if abs(observado-float(fila.roc_auc))>2e-5:
                raise AssertionError(f'Las predicciones de {codigo} no concuerdan con el resumen agregado.')
    print('Vectores OOF privados verificados:',len(resultados_oof))
else:
    print('No se localizaron las predicciones OOF privadas; los resultados agregados permanecen disponibles.')

In [ ]:
if resultados_oof:
    resultados_oof=dict(sorted(resultados_oof.items(),key=lambda t:roc_auc_score(y,t[1]['oof']),reverse=True))
    orden_pr=dict(sorted(resultados_oof.items(),key=lambda t:average_precision_score(y,t[1]['oof']),reverse=True))
    fig,ejes=plt.subplots(1,2,figsize=(14.5,5.5))
    for i,(nombre,datos) in enumerate(resultados_oof.items()):
        prob=datos['oof']; color=COLORES[i%len(COLORES)]
        fpr,tpr,_=roc_curve(y,prob)
        sel=np.unique(np.linspace(0,len(fpr)-1,min(2500,len(fpr)),dtype=int))
        fpr,tpr=fpr[sel],tpr[sel]
        prec,rec,_=precision_recall_curve(y,prob,drop_intermediate=True)
        sel=np.unique(np.linspace(0,len(prec)-1,min(2500,len(prec)),dtype=int))
        prec,rec=prec[sel],rec[sel]
        ejes[0].plot(fpr,tpr,color=color,alpha=1 if i==0 else .5,label=f'{traduccion_estrategias[nombre]} · {roc_auc_score(y,prob):.4f}')

    for i,(nombre,datos) in enumerate(orden_pr.items()):
        prec,rec,_=precision_recall_curve(y,datos['oof'],drop_intermediate=True)
        sel=np.unique(np.linspace(0,len(prec)-1,min(2500,len(prec)),dtype=int))
        color=COLORES[list(resultados_oof).index(nombre)%len(COLORES)]
        ejes[1].plot(rec[sel],prec[sel],color=color,alpha=1 if i==0 else .5,
                     label=f'{traduccion_estrategias[nombre]} · {average_precision_score(y,datos["oof"]):.4f}')
    ejes[0].plot([0,1],[0,1],ls='--',color='gray',label='Azar')
    ejes[0].set(title='Curvas ROC en predicciones OOF',xlabel='Tasa de falsos positivos',ylabel='Tasa de verdaderos positivos')
    ejes[1].axhline(np.mean(y),ls='--',color='gray',label='Prevalencia de fraude')
    ejes[1].set(title='Curvas precisión–exhaustividad',xlabel='Exhaustividad (sensibilidad)',ylabel='Precisión')
    for ax in ejes:ax.legend(fontsize=8,loc='best')
    cerrar_figura(fig,'15_curvas_roc_y_precision_exhaustividad')
    fig,ax=plt.subplots(figsize=(9,6))
    for i,(nombre,datos) in enumerate(resultados_oof.items()):
        tabla=pd.DataFrame({'real':y,'riesgo':datos['oof']})
        tabla['intervalo']=pd.qcut(tabla['riesgo'],q=15,duplicates='drop')
        grupo=tabla.groupby('intervalo',observed=True).agg(p_pred=('riesgo','mean'),p_real=('real','mean'))
        ax.plot(grupo['p_pred'],grupo['p_real'],marker='o',markersize=3,alpha=1 if i==0 else .5,label=traduccion_estrategias[nombre],color=COLORES[i])
    ax.plot([0,1],[0,1],color='black',ls='--',label='Calibración ideal')
    ax.set(title='Calibración observada por intervalos de riesgo OOF',xlabel='Probabilidad media estimada',ylabel='Frecuencia observada de fraude',xlim=(0,1),ylim=(0,1))
    ax.legend(fontsize=8)
    cerrar_figura(fig,'16_calibracion_oof')
else:
    print('Curvas y calibración omitidas por ausencia de vectores OOF privados.')

### 5.3 ROC AUC frente a tiempo de entrenamiento

Se comparan los valores medidos de AUC y los tiempos de ajuste del booster por fold. **Estos tiempos no incluyen la preparación de datos ni, para el híbrido, el ajuste de la regresión logística**, por lo que no representan el costo total de las arquitecturas. Su utilidad es ilustrar la relación entre complejidad incremental y ganancia predictiva.

In [ ]:
ruta_tiempos=RAIZ/'experiments'/'experiment018'/'resultados'/'tiempos_historicos.csv'
if ruta_tiempos.exists():
    eficiencia=pd.read_csv(ruta_tiempos).sort_values('roc_auc',ascending=False)
    fig,ax=plt.subplots(figsize=(12.5,6))
    sns.scatterplot(data=eficiencia,x='segundos_booster_por_fold',y='roc_auc',
                    hue='familia',hue_order=eficiencia.familia.drop_duplicates().tolist(),palette='dark',s=130,ax=ax)
    for r in eficiencia.itertuples():
        ax.annotate(r.nombre,(r.segundos_booster_por_fold,r.roc_auc),xytext=(5,6),textcoords='offset points',fontsize=8)
    ax.set(title='ROC AUC frente a tiempo del ajuste supervisado',xlabel='Segundos por fold: booster (no incluye preparación ni logística)',ylabel='ROC AUC OOF')
    ax.legend(title='Familia')
    cerrar_figura(fig,'20_roc_auc_vs_tiempo')
    display(eficiencia.round(5))

## Etapa 6. Incertidumbre: bootstrap emparejado

La diferencia $\Delta=\mathrm{AUC}(A)-\mathrm{AUC}(B)$ se calcula **sobre las mismas transacciones**. Un remuestreo de fraudes y no fraudes por separado mantiene el tamaño de cada clase y permite aproximar la variabilidad condicional de $\Delta$. En el experimento010 se utilizaron **600 réplicas** y un intervalo percentil del 95 %.

**Alcance.** Este procedimiento mantiene fijas las predicciones OOF; no reentrena los modelos en cada réplica y no corrige completamente el sesgo de haber seleccionado el ganador en los mismos datos, posibles transacciones dependientes ni los problemas de anidación del stacking. Por ello no se interpreta como prueba definitiva de superioridad fuera de muestra.

In [ ]:
bootstrap=pd.read_csv(DATOS_AGREGADOS/'bootstrap_experiment010.csv')
bootstrap=bootstrap.sort_values('diferencia_auc')
fig,ax=plt.subplots(figsize=(12,6.2))
for j,fila in enumerate(bootstrap.itertuples()):
    ax.plot([fila.ic95_inferior,fila.ic95_superior],[j,j],lw=3,color=COLORES[0])
    ax.scatter(fila.diferencia_auc,j,s=60,color=COLORES[3],zorder=3)
ax.axvline(0,linestyle='--',color='black',label='Sin diferencia de ROC AUC')
ax.set(yticks=np.arange(len(bootstrap)),yticklabels=bootstrap['estrategia'].tolist(),
       xlabel='Diferencia respecto de XGBoost individual (ROC AUC)',ylabel='',
       title='Intervalos percentiles del bootstrap emparejado (95 %)')
ax.legend()
cerrar_figura(fig,'17_bootstrap_comparacion_ensambles')
display(bootstrap.sort_values('roc_auc_oof',ascending=False).round(6))

## 7. Selección, decisión y entrega a Kaggle

En la comparación original el XGBoost individual obtuvo AUC OOF **0.946120** y la corrección de logits **0.946205**. La diferencia +0.000085 presentó IC emparejado 95 % [−0.000263,+0.000440], que incluye cero. La corrección logística fue la selección nominal por AUC, **no una mejora estadísticamente establecida**. Los resultados de una nueva búsqueda deben compararse con esta referencia, sin atribuirles puntuaciones históricas.

El CSV de Kaggle contiene probabilidades `id,is_fraud`. El umbral de fraude es una política operativa separada del ROC AUC.

In [ ]:
seleccionada='ensemble2_logit_offset'  # Selección histórica verificable por ROC AUC OOF.
if seleccionada in resultados_oof:
    prob=np.asarray(resultados_oof[seleccionada]['test'],dtype=float)
    if prob.shape!=(len(prueba),) or not np.isfinite(prob).all() or np.min(prob)<0 or np.max(prob)>1:
        raise ValueError('El vector de probabilidades para Kaggle no es válido')
    envio=plantilla_envio[[IDENTIFICADOR]].copy()
    envio[OBJETIVO]=prob
    assert envio[IDENTIFICADOR].equals(prueba[IDENTIFICADOR])
    ruta=SALIDA/'envio_final_kaggle.csv'
    envio.to_csv(ruta,index=False)
    display(envio.head(6))
    print('Archivo de Kaggle creado:',ruta,'| filas:',len(envio))
else:
    print('No se creó un envío: es necesario recuperar las predicciones de prueba del modelo seleccionado.')

### 7.1 ¿Cuándo emitir una alerta de fraude?

Las probabilidades pueden convertirse en decisiones mediante `predicción ≥ umbral`. Reducir falsos negativos suele incrementar falsos positivos. Se estudia el umbral convencional 0.50, el que maximiza F1 y el **mayor umbral compatible con 98 % de exhaustividad** sobre datos de selección, solo como objetivo ilustrativo. Se reportan VP, VN, FP, FN, sus porcentajes, F1, precisión, recall, exactitud y ROC AUC.

Los umbrales de evaluación cruzada se eligen en cuatro folds y se aplican al quinto; esto reduce optimismo pero no equivale a validación independiente ni resuelve dependencias por cliente.

In [ ]:
from sklearn.metrics import (confusion_matrix,f1_score,accuracy_score,precision_score,recall_score,
                             balanced_accuracy_score)
OBJETIVO_RECALL=0.98

def elegir_umbral(y_real,p,objetivo=OBJETIVO_RECALL):
    positivos=np.sort(np.asarray(p)[np.asarray(y_real)==1])
    if not len(positivos):raise ValueError('No hay observaciones fraudulentas')
    return float(positivos[max(0,int(np.floor((1-objetivo)*len(positivos))))])

def metricas_alerta(y_real,p,umbral,nombre,politica):
    b=np.asarray(p)>=umbral
    vn,fp,fn,vp=confusion_matrix(y_real,b,labels=[0,1]).ravel()
    return {'Estrategia':nombre,'Política':politica,'Umbral':umbral,
            'VP':int(vp),'FN':int(fn),'FP':int(fp),'VN':int(vn),
            'Recall':recall_score(y_real,b),'Precisión':precision_score(y_real,b,zero_division=0),
            'F1':f1_score(y_real,b,zero_division=0),'Exactitud':accuracy_score(y_real,b),
            'ROC AUC':roc_auc_score(y_real,p),'Tasa de falsas alarmas':fp/(fp+vn)}

predicciones={k:v['oof'] for k,v in resultados_oof.items()}
# Los resultados nuevos se incorporan si el propietario conserva el archivo OOF local.
archivo=RAIZ/'outputs'/'experiment017'/'full_699960'/'checkpoint_PRIVATE.npz'
if archivo.is_file():
    with np.load(archivo,allow_pickle=False) as z:
        for key,field in [('xgb_kmeans5_fijo','aug_fixed_oof'),('xgb_kmeans5_grilla30','aug_grid_oof')]:
            p=np.asarray(z[field])
            if len(p)==len(y) and np.isfinite(p).all():predicciones[key]=p
archivo30=RAIZ/'outputs'/'experiment017'/'depth30'/'checkpoint_depth30_PRIVATE.npz'
if archivo30.is_file():
    with np.load(archivo30,allow_pickle=False) as z:
        p=np.asarray(z['grid_depth30_oof'])
        if len(p)==len(y) and np.isfinite(p).all():predicciones['xgb_kmeans5_grilla36']=p
if predicciones:
    nombres={**traduccion_estrategias,'xgb_kmeans5_fijo':'XGBoost + K-means fijo',
             'xgb_kmeans5_grilla30':'XGBoost + K-means 30',
             'xgb_kmeans5_grilla36':'XGBoost + K-means 36'}
    registros=[];cruzados=[];predicciones_cruzadas={}
    pliegues=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEMILLA).split(np.zeros(len(y)),y))
    for codigo,p in predicciones.items():
        nombre=nombres.get(codigo,codigo)
        t=elegir_umbral(y,p)
        registros.append(metricas_alerta(y,p,.5,nombre,'Umbral 0,50'))
        pp,rr,tt=precision_recall_curve(y,p)
        f1=2*pp[:-1]*rr[:-1]/np.maximum(pp[:-1]+rr[:-1],1e-12)
        registros.append(metricas_alerta(y,p,float(tt[np.argmax(f1)]),nombre,'Máximo F1 OOF'))
        registros.append(metricas_alerta(y,p,t,nombre,'Recall 98 % OOF'))
        decisiones=np.zeros(len(y),dtype=bool)
        for tr,va in pliegues:
            decisiones[va]=p[va]>=elegir_umbral(y[tr],p[tr])
        predicciones_cruzadas[codigo]=decisiones
        vn,fp,fn,vp=confusion_matrix(y,decisiones,labels=[0,1]).ravel()
        cruzados.append({'Estrategia':nombre,'FN':fn,'FP':fp,
                         'Recall cruzado':vp/(vp+fn),'F1 cruzado':f1_score(y,decisiones),
                         'Exactitud cruzada':accuracy_score(y,decisiones),
                         'Falsas alarmas':fp/(vn+fp),'ROC AUC OOF':roc_auc_score(y,p),
                         'Umbral OOF':t,'Código':codigo})
    tabla_alerta=pd.DataFrame(registros)
    tabla_cruzada=pd.DataFrame(cruzados).sort_values(['Falsas alarmas','FN'])
    display(tabla_alerta[['Estrategia','Política','Umbral','VP','FN','FP','VN','Recall','Precisión','F1','Exactitud','ROC AUC']].round(6))
    display(tabla_cruzada.round(6))
    tabla_alerta.to_csv(SALIDA/'metricas_alerta.csv',index=False)
    tabla_cruzada.to_csv(SALIDA/'metricas_alerta_cruzadas.csv',index=False)
    # Matriz ilustrativa de la opción con menor tasa de alertas falsas.
    elegido=tabla_cruzada.iloc[0]
    matriz=confusion_matrix(y,predicciones_cruzadas[elegido['Código']],labels=[0,1])
    fig,ax=plt.subplots(figsize=(8,6))
    sns.heatmap(matriz,annot=anotaciones_matriz(matriz),fmt='',cmap='Blues',cbar=False,
       xticklabels=['Legítima predicha','Alerta de fraude'],
       yticklabels=['Legítima real','Fraude real'],ax=ax)
    ax.set(title='Matriz de confusión cruzada · objetivo de exhaustividad 98 %',
           xlabel='Predicción',ylabel='Clase real')
    cerrar_figura(fig,'21_matriz_confusion_umbral')
    fig,ejes=plt.subplots(1,2,figsize=(14,6))
    sns.barplot(data=tabla_cruzada,y='Estrategia',x='FN',color=COLORES[1],ax=ejes[0])
    sns.barplot(data=tabla_cruzada,y='Estrategia',x='Falsas alarmas',color=COLORES[0],ax=ejes[1])
    ejes[0].set(title='Fraudes no detectados',xlabel='Falsos negativos',ylabel='')
    ejes[1].set(title='Tasa de alertas falsas',xlabel='Falsas alarmas / transacciones legítimas',ylabel='')
    cerrar_figura(fig,'22_compromiso_errores_operativos')
else:
    archivo=RAIZ/'experiments'/'experiment017'/'results'/'alert_metrics_crossfold.csv'
    if archivo.is_file():
        display(pd.read_csv(archivo)[['modelo','FN_cv','FP_cv','recall_cv','f1_cv','accuracy_cv','roc_auc_oof']].round(6))
        print('Resumen agregado disponible; el gráfico de confusión requiere vectores OOF locales.')
    else:
        print('Sin predicciones OOF: no se inventan métricas binarias.')

### 7.2 Diagnósticos operativos: F1, precisión, recall y exactitud

La matriz de confusión informa conteos y porcentajes sobre el conjunto OOF. Estas figuras comparan políticas de decisión y la sensibilidad del F1 ante el umbral; un objetivo de recall del 98 % puede producir un volumen muy elevado de alertas falsas. Los resultados usan predicciones OOF reales, no probabilidades inventadas.

In [ ]:
if predicciones:
    modelo=elegido['Estrategia']
    p_operativo=np.asarray(predicciones[elegido['Código']],dtype=float)
    politicas_modelo=tabla_alerta[tabla_alerta['Estrategia']==modelo]
    fig,ejes=plt.subplots(2,2,figsize=(13.5,9))
    for ax,(col,titulo) in zip(ejes.ravel(),[('F1','F1'),('Precisión','Precisión'),
                                           ('Recall','Exhaustividad'),('Exactitud','Exactitud')]):
        tab=politicas_modelo.sort_values(col,ascending=False)
        sns.barplot(data=tab,x=col,y='Política',color=COLORES[0],ax=ax)
        ax.set(title=titulo+' según la política',xlabel=titulo,ylabel='',xlim=(0,1.02))
    cerrar_figura(fig,'23_metricas_f1_precision_recall_accuracy')
    pp,rr,tt=precision_recall_curve(y,p_operativo,drop_intermediate=True)
    f1=2*pp[:-1]*rr[:-1]/np.maximum(pp[:-1]+rr[:-1],1e-12)
    selec=np.unique(np.linspace(0,len(tt)-1,min(2500,len(tt)),dtype=int))
    fig,ax=plt.subplots(figsize=(12,5.7))
    ax.plot(np.maximum(tt[selec],1e-6),rr[:-1][selec],color=COLORES[1],alpha=.5,label='Exhaustividad')
    ax.plot(np.maximum(tt[selec],1e-6),pp[:-1][selec],color=COLORES[2],alpha=.5,label='Precisión')
    ax.plot(np.maximum(tt[selec],1e-6),f1[selec],color=COLORES[0],alpha=1,lw=2.4,label='F1')
    ax.axvline(tt[np.argmax(f1)],color=COLORES[3],ls='--',label='Máximo F1')
    ax.axvline(float(elegido['Umbral OOF']),color=COLORES[4],ls='--',label='Objetivo 98 %')
    ax.set(xscale='log',title='F1, precisión y exhaustividad según el umbral',
           xlabel='Umbral (escala logarítmica)',ylabel='Métrica (0 a 1)',ylim=(0,1.03))
    ax.legend()
    cerrar_figura(fig,'24_f1_precision_recall_vs_umbral')
    fig,ejes=plt.subplots(1,2,figsize=(15,6))
    f1_ranking=tabla_cruzada.sort_values('F1 cruzado',ascending=False)
    sns.barplot(data=f1_ranking,x='F1 cruzado',y='Estrategia',color=COLORES[0],ax=ejes[0])
    ejes[0].set(title='F1 con umbral seleccionado en otros folds',xlabel='F1 cruzado',ylabel='')
    alertas_ranking=tabla_cruzada.sort_values('FP',ascending=True)
    sns.barplot(data=alertas_ranking,x='FP',y='Estrategia',color=COLORES[1],ax=ejes[1])
    ejes[1].set(title='Alertas sobre operaciones legítimas',xlabel='Falsos positivos',ylabel='')
    cerrar_figura(fig,'25_f1_y_alertas_por_modelo')

else:
    print('Se requieren predicciones OOF privadas para las curvas de umbral y F1.')

## Etapa 8. Limitaciones y conclusiones

**Conclusiones principales.** La primera mejora metodológica fue evaluar probabilidades mediante cinco particiones estratificadas y conservar predicciones OOF. La no linealidad justifica comparar la logística con árboles. El uso de PCA no garantiza conservar la información útil sobre la clase, y las nuevas distancias a clusters aportaron ganancias marginales o negativas. El ensamble más competitivo utilizó la regresión logística como punto de partida y XGBoost como corrector de logits, pero no demostró superioridad estadística sobre XGBoost individual.

**Limitaciones.** La selección de hiperparámetros y del mejor ensamble utiliza datos de desarrollo; el stacking reutiliza predicciones OOF en un segundo nivel no completamente anidado. El bootstrap condicional no reentrena modelos ni corrige esas fuentes de optimismo. Las observaciones de fraude proceden de un conjunto sintético de Kaggle, y no se dispone aquí de información suficiente sobre independencia por cliente, evolución temporal o desempeño sobre un conjunto de prueba privado etiquetado.

**Reproducción.** El modo predeterminado lee el conjunto original y resultados ya calculados. Los 19 candidatos originales, los ajustes de hiperparámetros y los experimentos de clustering se conservan en los notebooks y experimentos numerados del repositorio; no se repiten automáticamente para evitar costo computacional innecesario. Todas las figuras nuevas se guardan en español bajo `outputs/informe_final_es/figuras/`.

In [ ]:
catalogo=pd.DataFrame([{'Figura':p.stem.replace('_',' ').capitalize(), 'Archivo':str(p.relative_to(RAIZ)) if p.is_relative_to(RAIZ) else str(p)} for p in sorted(FIGURAS.glob('*.png'))])
display(catalogo)
(SALIDA/'catalogo_figuras.csv').parent.mkdir(parents=True,exist_ok=True)
catalogo.to_csv(SALIDA/'catalogo_figuras.csv',index=False)
resumen_final={
 'fecha':'2026-11-24', 'observaciones_entrenamiento':len(entrenamiento),
 'observaciones_prueba':len(prueba),'n_particiones':N_PARTICIONES,
 'mejor_individual':'XGBoost (profundidad 5, tasa 0.05)',
 'seleccion_nominal':'Regresión logística corregida con XGBoost',
 'auc_mejor_individual':0.9461201492641279,
 'auc_seleccionado':0.9462047549947218,
 'delta_auc':8.460573059387855e-05,
 'ic95_delta':[-0.00026258745106241544,0.00043972556755127727],
 'metodo_ci':'Bootstrap emparejado estratificado, 600 réplicas, condicional sobre OOF',
 'se_entrenaron_modelos_ahora':bool(CORRER_BUSQUEDA_DE_EJEMPLO),
 'hay_predicciones_privadas':bool(resultados_oof),
 'nota':'Se reutilizan resultados históricos; no equivale a validación independiente.'}
(SALIDA/'resumen_final.json').write_text(json.dumps(resumen_final,indent=2,ensure_ascii=False),encoding='utf8')
print('Figuras:',len(catalogo),'| ruta:',FIGURAS.resolve())
print('Resultado: selección nominal por AUC OOF, sin ventaja estadística demostrada.')